# 04 — Model Arena and the decision-support workflow

This notebook reproduces the Model Arena benchmark and a human-in-the-loop agent run. The benchmark compares a human-selected model against the validation-selected model on the same untouched holdout. The agent calls vetted analytics services and writes grounded recommendations; it never executes user code or modifies source data.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "backend").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "backend"))

import pandas as pd

from app.database.base import SessionLocal, init_db
from app.services.data_service import ensure_demo_dataset, get_context

init_db()
db = SessionLocal()
dataset = ensure_demo_dataset(db)
ctx = get_context(dataset.id, db)
print(f"Benchmarking on {dataset.name}")

## Revenue forecast benchmark

A human chooses Linear Regression. The AI selection uses expanding-window validation on the training data only. Both models are then evaluated on the identical final holdout, so the holdout cannot influence model selection.

In [ ]:
from app.services.benchmark_service import run_benchmark

revenue_benchmark = run_benchmark(
    ctx,
    task="revenue_forecast",
    human_model="linear_regression",
    test_ratio=0.2,
    db=db,
)
print(f"Human: {revenue_benchmark['human']['label']} | AI: {revenue_benchmark['ai']['label']}")
pd.DataFrame(revenue_benchmark["comparison_table"])

In [ ]:
candidates = pd.json_normalize(revenue_benchmark["candidates"])
candidates.sort_values("validation_mae").head(10)

## Churn benchmark

Classification metrics use a stratified holdout. Recency is excluded from the churn features because the label is defined from inactivity, which would otherwise leak the target.

In [ ]:
churn_benchmark = run_benchmark(
    ctx,
    task="churn_classification",
    human_model="logistic_regression",
    test_ratio=0.25,
    db=db,
)
pd.DataFrame(churn_benchmark["comparison_table"])

## Human-in-the-loop agent proposal

The proposal lists the available tasks, evidence of data quality and the human checkpoint. It is deterministic and does not call an LLM. Executing below is the notebook equivalent of approving the proposal in the UI.

In [ ]:
from app.agents.runner import available_tasks, create_run, execute_agent_run

objective = "Prioritize the most important revenue and customer-retention decisions."
available = available_tasks(ctx)
selected = [task for task in ["revenue", "customers"] if task in available]
run = create_run(db, ctx, mode="hitl", objective=objective, selected_tasks=selected)

print(f"Available tasks: {available}")
print(f"Selected tasks: {run.selected_tasks}")
print(f"Status: {run.status}")
pd.json_normalize(run.proposal["proposed_tasks"])[["label", "available", "selected", "reason"]]

In [ ]:
run.status = "running"
db.commit()
execute_agent_run(run.id)

from app.agents.runner import run_to_dict
from app.models import AgentRun

db.expire_all()
finished = db.get(AgentRun, run.id)
payload = run_to_dict(finished)
print(f"Status: {payload['status']} | LLM narrative: {payload['llm_used']} | Seconds: {payload['duration_seconds']}")
pd.DataFrame(payload["steps"])[["label", "status", "summary"]]

## Decision output

Recommendations are ranked from computed evidence. Expected impacts are directional unless the data supports a quantified estimate. The optional LLM only writes a narrative around these deterministic results.

In [ ]:
decisions = pd.DataFrame(payload["result"].get("decisions", []))
decisions.reindex(columns=["priority", "area", "title", "action", "source"])

In [ ]:
print(payload["result"].get("executive_summary", "No executive summary was produced."))
print("\nDecision basis:", payload["result"].get("decision_basis"))

In [ ]:
db.close()